## Data Cleaning for Ag slab w/ [C, H, O] adsorbates
##### DropBox or Box --> filtered folder names

In [53]:
import os
import time
from dotenv import load_dotenv
from pymatgen.io.vasp.inputs import Incar

load_dotenv(override=True)

# USER INPUT HERE: set cache folder for the dataset being cleaned
# CACHE = "cache/dropbox_O2DissociationSAA_2"
CACHE = "cache/box_EO_Project_reactivity"

### Get files from DropBox

In [2]:
import dropbox
from dropbox.files import SharedLink

DB_TOKEN = os.getenv("DROPBOX_TOKEN")
DB_APP_KEY = os.getenv("APP_KEY")
DB_APP_SECRET = os.getenv("APP_SECRET")
DB_REFRESH_TOKEN = os.getenv("DROPBOX_REFRESH_TOKEN")

One-time DropBox refresh token authentication (don't need to rerun this)

In [3]:
from dropbox.oauth import DropboxOAuth2FlowNoRedirect

auth_flow = DropboxOAuth2FlowNoRedirect(
    consumer_key=DB_APP_KEY,
    consumer_secret=DB_APP_SECRET,
    token_access_type="offline",
)

print(auth_flow.start())

https://www.dropbox.com/oauth2/authorize?response_type=code&client_id=jwuqsuv0crfxtab&token_access_type=offline


In [ ]:
code = "" 

oauth_result = auth_flow.finish(code)
print("Refresh token:", oauth_result.refresh_token)
print("Access token:", oauth_result.access_token)

In [4]:
# access dropbox API

dbx = dropbox.Dropbox(
    oauth2_refresh_token=DB_REFRESH_TOKEN,
    app_key=DB_APP_KEY,
    app_secret=DB_APP_SECRET,
)

ROOT_PATH = SharedLink(url="https://www.dropbox.com/scl/fo/qpg1zuo3g7vb3il1wmqy3/AA4wERzz28lJhYTCAcSEvqk?dl=0")

In [ ]:
# fetch list of all subfolders in 1st level of DropBox folder

results = []
def list_folder():
    res = dbx.files_list_folder(path="", shared_link=ROOT_PATH)
    
    while True:
        for entry in res.entries:
            results.append({
                "name": entry.name,
                "type": entry.__class__.__name__
            })
        if not res.has_more:
            break
        res = dbx.files_list_folder_continue(res.cursor)

list_folder()
print(f"Total items: {len(results)}")

# write txt file of folder names
with open(f"{CACHE}/raw_foldernames.txt", "w", encoding="utf-8") as f:
    for entry in results:
        if entry["type"] == "FolderMetadata":
            f.write(entry["name"] + "\n")

print(f"Saved to {CACHE}/raw_foldernames.txt")

### Get files from Box

In [54]:
from box_sdk_gen import BoxClient, BoxDeveloperTokenAuth
load_dotenv(override=True)

BOX_DEV_TOKEN = os.getenv("DEV_TOKEN")
BOX_FOLDER_IDS = ["376580648972", "376581493469"]
# EO_Project_reactivity, EO_Project_reactivity_part2

auth = BoxDeveloperTokenAuth(BOX_DEV_TOKEN)
client = BoxClient(auth)

In [36]:
# check number of first-level subfolders

for folder_id in BOX_FOLDER_IDS:
    info = client.folders.get_folder_by_id(folder_id, fields=["item_collection", "name"])
    print(f"folder id: {folder_id}, info.name, folder count: {info.item_collection.total_count}")

folder id: 376580648972, info.name, folder count: 295
folder id: 376581493469, info.name, folder count: 76


In [ ]:
# get all folder names and write list to cache
allnames = set()

for folder_id in BOX_FOLDER_IDS:
    offset = 0
    limit = 1000

    while True:
        items = client.folders.get_folder_items(
            folder_id,
            limit=limit,
            offset=offset,
        )

        for item in items.entries:
            if item.type == "folder":
                allnames.add(item.name)

        offset += len(items.entries)
        if offset >= items.total_count:
            break

with open(f"{CACHE}/raw_foldernames.txt", "w", encoding="utf-8") as f:
    for name in sorted(allnames):
        f.write(name + "\n")

print(f"Wrote {len(allnames)} folder names to {CACHE}/raw_foldernames.txt")

Saved 369 folder names


## Filter through folder names

In [37]:
# read in raw folder names list from cache

with open(f"{CACHE}/raw_foldernames.txt", "r", encoding="utf-8") as f:
    foldernames = [line.strip() for line in f]

# filter out obviously bad data via substring matching
exclude = {
    "wrong",
    "toomanykpoints",
    "oldbad",
    "exploded",
        }

foldernames = [n for n in foldernames if not any(term in n for term in exclude)]
print(f"Total items: {len(foldernames)}")

Total items: 369


In [38]:
# filter out elements and calculation types we don't want
exclude = {
    "Al", "Au", "Co", "Cr", "Cu", "Fe", "Ga", "In", "Ir", "Mg", "Mn", "Mo", "Pd", "Pt", "Re", "Rh", "Ru", "Sc", "Sn", "Ti", "V", "W", "Zn",
    "minhop", "CL", "CL_JS", "Bader", "Bare", "bare", "Bulk", "Gas", "gas", "TS", "spin"
}

folders = []

# if trying to exclude Ni, set False: 
NI = True
# include names w/ "Ni0" substring (means zero Ni)
for name in foldernames:
    if not NI and "Ni" in name and "Ni0" not in name:
        continue
    if any(term in name for term in exclude):
        continue
    folders.append(name)

foldernames = folders

print(f"Total items: {len(foldernames)}")

Total items: 228


### for DropBox

In [19]:
# check if OUTCAR exists and if so, get elements from POSCAR

elements_map = {}
irregular_poscars = {}
no_poscar = []
no_outcar = []
failed_folders = {}

for folder in foldernames:
    folder_path = f"/{folder}"
    try:
        res = dbx.files_list_folder(path=folder_path, shared_link=ROOT_PATH)
        names = {e.name for e in res.entries}
    except dropbox.exceptions.ApiError as e:
        failed_folders[folder] = str(e)
        continue

    if "OUTCAR" not in names:
        no_outcar.append(folder)
        continue

    try:
        _, dres = dbx.sharing_get_shared_link_file(url=ROOT_PATH.url, path=f"{folder_path}/POSCAR")
        lines = dres.content.decode("utf-8", errors="ignore").splitlines()
        tokens = lines[5].split()
        if all(t.isalpha() for t in tokens):
            elements_map[folder] = set(tokens)
        else:
            irregular_poscars[folder] = lines[:5]
    except dropbox.exceptions.ApiError as e:
        failed_folders[folder] = str(e)

    time.sleep(0.2)

### for Box

In [39]:
# get all folder names+ids via Box API and store in set

subfolder_map = {}  # name -> folder_id
for folder_id in BOX_FOLDER_IDS:
    offset = 0
    limit = 1000
    while True:
        items = client.folders.get_folder_items(folder_id, limit=limit, offset=offset)
        for item in items.entries:
            if item.type == "folder":
                subfolder_map[item.name] = item.id
        offset += len(items.entries)
        if offset >= items.total_count:
            break

In [42]:
# check if OUTCAR exists and if so, get elements from POSCAR
from box_sdk_gen import BoxAPIError

elements_map = {}
irregular_poscars = {}
no_outcar = []
no_poscar = []
failed_folders = {}

for folder, folder_id in subfolder_map.items():
    try:
        items = client.folders.get_folder_items(folder_id, limit=1000)
        names = {e.name: e.id for e in items.entries}
    except BoxAPIError as e:
        failed_folders[folder] = str(e)
        continue

    if "OUTCAR" not in names:
        no_outcar.append(folder)
        continue

    if "POSCAR" not in names:
        no_poscar.append(folder)
        continue

    try:
        file_stream = client.downloads.download_file(names["POSCAR"])
        content = file_stream.read()
        lines = content.decode("utf-8", errors="ignore").splitlines()
        tokens = lines[5].split()
        if all(t.isalpha() for t in tokens):
            elements_map[folder] = set(tokens)
        else:
            irregular_poscars[folder] = lines[:5]
    except BoxAPIError as e:
        failed_folders[folder] = str(e)
    time.sleep(0.2)

**continue**

In [43]:
print(f"""
    elements_map: {len(elements_map)} items
    irregular_poscars: {len(irregular_poscars)} items
    no_outcar: {len(no_outcar)} items
    no_poscar: {len(no_poscar)} items
    failed_folders: {len(failed_folders)} items
    """)


    elements_map: 313 items
    irregular_poscars: 0 items
    no_outcar: 55 items
    no_poscar: 1 items
    failed_folders: 0 items
    


In [23]:
print(irregular_poscars)
print(elements_map)

{'1OAg': [' O Ag ', ' 1.0000000000000000', '     8.6296027574400007    0.0000000000000000    0.0000000000000000', '     4.3148013787200004    7.4734547894199999    0.0000000000000000', '     0.0000000000000000    0.0000000000000000   23.4868010454999983'], '2ONi': [' O Ag Ni Ag ', ' 1.0000000000000000', '     8.6296027574400007    0.0000000000000000    0.0000000000000000', '     4.3148013787200004    7.4734547894199999    0.0000000000000000', '     0.0000000000000000    0.0000000000000000   23.4868010454999983'], 'O2AdsNi': [' O Ag Ni Ag ', ' 1.0000000000000000', '     8.6296027574400007    0.0000000000000000    0.0000000000000000', '     4.3148013787200004    7.4734547894199999    0.0000000000000000', '     0.0000000000000000    0.0000000000000000   23.4868010454999983'], '2O_subNi': [' O Ag Ni Ag ', ' 1.0000000000000000', '     8.6296027574400007    0.0000000000000000    0.0000000000000000', '     4.3148013787200004    7.4734547894199999    0.0000000000000000', '     0.00000000000000

In [44]:
# filter through regular POSCARs by element
allowed = {"Ag", "C", "H", "O", "Ni"}
clean_map = {k: v for k, v in elements_map.items() if v.issubset(allowed)}

print(f"Total items: {len(clean_map)}")

Total items: 313


In [ ]:
# inspect irregular POSCARs

for folder, lines in irregular_poscars.items():
    print(folder)
    print("\n".join(lines))
    print("-"*10)

In [24]:
# add irregular poscar elements to dict from file header 

elements_map_firstline = {}
still_irregular = {}

for folder, lines in irregular_poscars.items():
    tokens = lines[0].split()

    if tokens and all(t.isalpha() for t in tokens):
        elements_map_firstline[folder] = set(tokens)
    else:
        still_irregular[folder] = lines

print(f"Recovered {len(elements_map_firstline)} POSCARs")
print(f"Still irregular: {len(still_irregular)}")

Recovered 46 POSCARs
Still irregular: 2


In [28]:
allowed = {"Ag", "C", "H", "O", "Ni"}

clean_firstline_map = {
    k: v
    for k, v in elements_map_firstline.items()
    if v.issubset(allowed)
}

elements_map.update(clean_firstline_map)
print(len(elements_map), " configs kept.")

391  configs kept.


In [45]:
# last round of misc. filtering from manual inspection

exclude = {"smear", "DOS"}
clean_map = {
    k: v for k, v in clean_map.items()
    if v.issubset(allowed) and not any(sub in k for sub in exclude)
}

# delete folder if folder_good exists
suffixed = {k[:-5] for k in clean_map if k.endswith("_good")}
clean_map = {k: v for k, v in clean_map.items() if k not in suffixed}

print(f"Total items: {len(clean_map)}")

Total items: 310


In [46]:
# write txt file of cleaned folder names
with open(f"{CACHE}/withNi_cleaned_foldernames.txt", "w", encoding="utf-8") as f:
    for entry in clean_map:
        f.write(entry + "\n")

print(f"Saved to {CACHE}/withNi_cleaned_foldernames.txt")

Saved to cache/box_EO_Project_reactivity/withNi_cleaned_foldernames.txt


### for DropBox

In [48]:
# get INCARs for clean folders

incar_map = {}
missing_incars = []

for folder in clean_map:
    try:
        _, res = dbx.sharing_get_shared_link_file(url=ROOT_PATH.url, path=f"/{folder}/INCAR")
        incar_map[folder] = Incar.from_str(res.content.decode("utf-8", errors="ignore"))
    except dropbox.exceptions.ApiError:
        missing_incars.append(folder)
    time.sleep(0.2)

print(f"Total items: {len(incar_map)}, Missing INCARs: {len(missing_incars)}")

Total items: 7, Missing INCARs: 303


### for Box

In [55]:
# get INCARs for clean folders

incar_map = {}
missing_incars = []

for folder in clean_map:
    folder_id = subfolder_map[folder]
    try:
        items = client.folders.get_folder_items(folder_id, limit=1000)
        names = {e.name: e.id for e in items.entries}
        if "INCAR" not in names:
            missing_incars.append(folder)
            continue
        file_stream = client.downloads.download_file(names["INCAR"])
        content = file_stream.read()
        incar_map[folder] = Incar.from_str(content.decode("utf-8", errors="ignore"))
    except BoxAPIError:
        missing_incars.append(folder)
    time.sleep(0.2)

print(f"Total items: {len(incar_map)}, Missing INCARs: {len(missing_incars)}")

Total items: 310, Missing INCARs: 0


**continue**

In [56]:
# filter out DFT calculations other than relaxations

def is_relaxation(incar):
    return (
        incar.get("IBRION", None) in {1, 2, 3}
        and incar.get("ISPIN", 1) == 1
        and incar.get("NSW", 0) > 5    # reject runs with <5 ionic steps
        and "DDR" not in incar         # reject runs with dimer calculation tags
        and "ICHAIN" not in incar
    )

filtered_incar_map = {
    k: v for k, v in incar_map.items()
    if is_relaxation(v)
}

print(f"Total items: {len(filtered_incar_map)}")

Total items: 197


In [57]:
# write txt file of cleaned and selected folder names (relaxations)
with open(f"{CACHE}/withNi_selected_foldernames.txt", "w", encoding="utf-8") as f:
    for entry in filtered_incar_map:
        f.write(entry + "\n")

print(f"Saved to {CACHE}/withNi_selected_foldernames.txt")

Saved to cache/box_EO_Project_reactivity/withNi_selected_foldernames.txt


continue data preparation in `ag-data-selection.ipynb`

### If needed for another round of filtering: read in cleaned folder names list from cache 

In [ ]:
with open(f"{CACHE}/selected_foldernames.txt", "r", encoding="utf-8") as f:
    foldernames = [line.strip() for line in f]

with open("cache/screened_foldernames.txt", "r", encoding="utf-8") as f:
    screenednames =  [line.strip() for line in f]

# check edge cases (runs that made it into `selected` but not `screened`)

foldernames = [v for v in foldernames if not any(sub in v for sub in exclude)]
edgecases = [v for v in foldernames if v not in screenednames]

selected_len = len(foldernames)
screened_len = len(screenednames)

print(f"Selected items: {selected_len}   Screened items: {screened_len}   Edge cases: {selected_len - screened_len}")